# Grouped 5-fold cross-validation + final holdout

Runs both models over the work-grouped folds from `04b_grouped_splits`, then trains each on everything except the holdout and scores the holdout **once**.

What this fixes relative to notebooks 05–07:

1. **No work-level leakage.** Movements of the same piece stay on one side of every boundary. The single-split numbers (LSTM 0.496, CNN 0.785) were measured with ~14% of test sharing a work with train, so expect these to come in lower. Lower and honest beats higher and leaky.
2. **Error bars.** Five folds give a standard deviation, so "the CNN reads style better than the LSTM" becomes a claim with evidence behind it instead of a single noisy number.
3. **Augmentation applied in memory, per fold.** Notebook 04 wrote pitch-shifted copies to disk for its own train set. Reusing those here would put an augmented copy of a piece in one fold and its original in another. Instead the shifts are recreated at load time, from `SHIFTS` below, using the same rule as notebook 04 — so a shifted copy can only ever exist inside the fold that owns its original.

**Progress tracking.** This is a twelve-training run. `nbconvert` buffers cell output and only flushes when the whole notebook finishes, so `print` gives no visibility while it is going. Everything therefore also appends to `results/cv_progress.log`, flushed on every write — `tail -f results/cv_progress.log` shows live progress, one line per epoch.

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

LSTM_DIR = Path("../data/processed/lstm")
CNN_DIR = Path("../data/processed/cnn")
GROUP_DIR = Path("../data/splits_grouped")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)
PROGRESS = RESULTS_DIR / "cv_progress.log"

COMPOSERS = ["bach", "beethoven", "chopin", "mozart"]
LABEL = {c: i for i, c in enumerate(COMPOSERS)}

# Same augmentation policy as notebook 04, recreated in memory rather than on disk.
SHIFTS = {"bach": [], "beethoven": [-1, 1], "mozart": [-1, 1], "chopin": [-2, -1, 1, 2]}

MAX_NOTES = 500
PITCHES = 88
MAX_STEPS = 512
BATCH = 32
N_FOLDS = 5

# CV runs 10 trainings before the two final ones, so the per-fold budget is tighter
# than the 100 epochs notebooks 05/06 use. The CNN reached val 0.81 by epoch 43 there,
# so 60 with patience 8 leaves headroom without a multi-hour run.
EPOCHS_CV = 60
PATIENCE_CV = 8


def log(msg):
    """Print and append to results/cv_progress.log, flushed immediately so the run
    can be followed with `tail -f` while nbconvert holds cell output."""
    line = f"{datetime.now(timezone.utc).strftime('%H:%M:%S')} {msg}"
    print(line)
    with open(PROGRESS, "a") as fh:
        fh.write(line + "\n")
        fh.flush()


PROGRESS.write_text("")  # fresh log per run
log(f"start, tensorflow {tf.__version__}")

folds_df = pd.read_csv(GROUP_DIR / "folds.csv")
holdout_df = pd.read_csv(GROUP_DIR / "holdout.csv")
log(f"folds {len(folds_df)} files, holdout {len(holdout_df)} files")

## Cache the originals once

Every fold re-reads the same files, so the cropped originals are loaded once into memory and the pitch shifts are applied as array operations on top. Cropping happens on the time axis and shifting on the pitch axis, so the two commute — shifting a cropped roll gives the same result as cropping a shifted one.

In [ ]:
def read_lstm(composer, filename):
    z = np.load(LSTM_DIR / composer / (Path(filename).stem + ".npz"))
    n = min(len(z["pitch"]), MAX_NOTES)
    pitch = np.zeros(MAX_NOTES, dtype=np.int32)
    cont = np.zeros((MAX_NOTES, 3), dtype=np.float32)
    if n:
        off = z["offset"][:n]
        pitch[:n] = z["pitch"][:n]
        cont[:n, 0] = np.clip(z["duration"][:n], 0, 4) / 4.0
        cont[:n, 1] = z["velocity"][:n] / 127.0
        cont[:n, 2] = np.clip(np.diff(off, prepend=off[0]), 0, 4) / 4.0
    return pitch, cont, n


def read_cnn(composer, filename):
    a = np.load(CNN_DIR / composer / (Path(filename).stem + ".npy"))
    x = np.zeros((PITCHES, MAX_STEPS), dtype=np.uint8)
    t = min(a.shape[1], MAX_STEPS)
    x[:, :t] = a[:, :t]
    return x


all_files = pd.concat([folds_df, holdout_df], ignore_index=True)
t0 = time.time()
LSTM_CACHE = {(r.composer, r.filename): read_lstm(r.composer, r.filename)
              for r in all_files.itertuples()}
CNN_CACHE = {(r.composer, r.filename): read_cnn(r.composer, r.filename)
             for r in all_files.itertuples()}
log(f"cached {len(LSTM_CACHE)} files in {time.time() - t0:.0f}s, "
    f"cnn cache ~{sum(a.nbytes for a in CNN_CACHE.values()) / 1e6:.0f} MB")

In [ ]:
def shift_lstm(cached, shift):
    """Transpose a note sequence. Real notes clip to 1..127 so a transposed-down
    note can never land on 0, which the Embedding treats as padding."""
    pitch, cont, n = cached
    if shift == 0:
        return pitch, cont
    out = pitch.copy()
    out[:n] = np.clip(pitch[:n].astype(np.int32) + shift, 1, 127)
    return out, cont


def shift_cnn(roll, shift):
    """Transpose a piano roll along the pitch axis, zero-filling — same as
    augment_cnn_file in notebook 04."""
    if shift == 0:
        return roll
    out = np.zeros_like(roll)
    if shift > 0:
        out[shift:, :] = roll[:PITCHES - shift, :]
    else:
        out[:PITCHES + shift, :] = roll[-shift:, :]
    return out


def build_arrays(df, kind, augment):
    """(dataframe, 'lstm'|'cnn', augment) -> (X, y). Augmented rows are added only
    for the composers listed in SHIFTS, and only when augment=True."""
    rows = [(r.composer, r.filename, 0) for r in df.itertuples()]
    if augment:
        rows += [(r.composer, r.filename, s) for r in df.itertuples()
                 for s in SHIFTS[r.composer]]
    y = np.array([LABEL[c] for c, _, _ in rows])
    if kind == "lstm":
        pairs = [shift_lstm(LSTM_CACHE[(c, f)], s) for c, f, s in rows]
        X = [np.stack([p for p, _ in pairs]), np.stack([q for _, q in pairs])]
    else:
        X = np.stack([shift_cnn(CNN_CACHE[(c, f)], s) for c, f, s in rows])[..., None]
    return X, y


# Sanity check: transposing by +2 must move every real note up by exactly 2.
_c, _f = folds_df.iloc[0][["composer", "filename"]]
_p0, _, _n = LSTM_CACHE[(_c, _f)]
_p2, _ = shift_lstm(LSTM_CACHE[(_c, _f)], 2)
assert (_p2[:_n] == np.clip(_p0[:_n] + 2, 1, 127)).all()
assert (_p2[_n:] == 0).all(), "padding must stay zero after a shift"
log("shift check ok")

## Models

Identical architectures to notebooks 05 and 06, so any difference here comes from the split and not from the model. Both carry the fixes those notebooks documented: the LSTM embeds pitch and clips gradients; the CNN has no BatchNorm.

In [ ]:
def build_lstm():
    pitch_in = tf.keras.layers.Input((MAX_NOTES,), dtype="int32", name="pitch")
    cont_in = tf.keras.layers.Input((MAX_NOTES, 3), name="continuous")
    e = tf.keras.layers.Embedding(128, 16, mask_zero=True)(pitch_in)
    x = tf.keras.layers.Concatenate()([e, cont_in])
    x = tf.keras.layers.LSTM(128, return_sequences=True)(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.LSTM(64)(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(4, activation="softmax")(x)
    m = tf.keras.Model([pitch_in, cont_in], out)
    m.compile(optimizer=tf.keras.optimizers.Adam(5e-4, clipnorm=1.0),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


def build_cnn():
    def block(x, filters, drop):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.MaxPooling2D(2)(x)
        return tf.keras.layers.Dropout(drop)(x)

    inp = tf.keras.layers.Input((PITCHES, MAX_STEPS, 1))
    x = tf.keras.layers.Rescaling(1 / 127.0)(inp)
    x = block(x, 32, 0.2)
    x = block(x, 64, 0.2)
    x = block(x, 128, 0.3)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    out = tf.keras.layers.Dense(4, activation="softmax")(x)
    m = tf.keras.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m


BUILD = {"lstm": build_lstm, "cnn": build_cnn}

In [ ]:
class LogProgress(tf.keras.callbacks.Callback):
    """One flushed log line per epoch, so a stalled run is visible immediately
    instead of at the end of the notebook."""

    def __init__(self, tag, total_epochs):
        super().__init__()
        self.tag = tag
        self.total = total_epochs
        self.t0 = None

    def on_train_begin(self, logs=None):
        self.t0 = time.time()

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        elapsed = time.time() - self.t0
        per_epoch = elapsed / (epoch + 1)
        eta = per_epoch * (self.total - epoch - 1)
        log(f"  {self.tag} epoch {epoch + 1}/{self.total} "
            f"loss {logs.get('loss', float('nan')):.3f} "
            f"acc {logs.get('accuracy', float('nan')):.3f} "
            f"val_acc {logs.get('val_accuracy', float('nan')):.3f} "
            f"[{per_epoch:.0f}s/epoch, eta {eta / 60:.0f}m]")


def score(y_true, y_pred):
    p_mac, r_mac, f_mac, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    p, r, f1, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=range(4), zero_division=0)
    return {
        "accuracy": float((y_pred == y_true).mean()),
        "precision_macro": float(p_mac),
        "recall_macro": float(r_mac),
        "f1_macro": float(f_mac),
        "per_class": {c: {"precision": float(p[i]), "recall": float(r[i]),
                          "f1": float(f1[i]), "support": int(sup[i])}
                      for i, c in enumerate(COMPOSERS)},
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=range(4)).tolist(),
    }


def class_weights_for(df):
    """From the unaugmented composer counts, matching notebooks 05/06."""
    y = df["composer"].map(LABEL).to_numpy()
    w = compute_class_weight("balanced", classes=np.arange(4), y=y)
    return dict(enumerate(w))

## Cross-validation

Early stopping uses the fold's own validation set, which makes each fold score mildly optimistic — the stopping epoch is chosen on the data being scored. Properly removing that needs nested CV, which triples the runtime for a correction smaller than the fold-to-fold spread we are trying to measure. Flagged rather than fixed, and it applies equally to both models so the comparison stays fair.

In [ ]:
def run_cv(kind):
    results = []
    for k in range(N_FOLDS):
        tr_df = folds_df[folds_df["fold"] != k]
        va_df = folds_df[folds_df["fold"] == k]
        X_tr, y_tr = build_arrays(tr_df, kind, augment=True)
        X_va, y_va = build_arrays(va_df, kind, augment=False)
        log(f"{kind} fold {k}: training on {len(y_tr)} rows, validating on {len(y_va)}")

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(SEED + k)
        model = BUILD[kind]()
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=PATIENCE_CV,
                restore_best_weights=True, verbose=0),
            LogProgress(f"{kind} f{k}", EPOCHS_CV),
        ]
        t0 = time.time()
        hist = model.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                         epochs=EPOCHS_CV, batch_size=BATCH,
                         class_weight=class_weights_for(tr_df),
                         callbacks=callbacks, verbose=0)

        y_pred = model.predict(X_va, verbose=0).argmax(axis=1)
        s = score(y_va, y_pred)
        s["fold"] = k
        s["epochs_run"] = len(hist.history["loss"])
        s["best_epoch"] = int(np.argmax(hist.history["val_accuracy"])) + 1
        s["seconds"] = round(time.time() - t0)
        results.append(s)
        log(f"{kind} fold {k} DONE: acc {s['accuracy']:.3f} macro-F1 {s['f1_macro']:.3f} "
            f"({s['epochs_run']} epochs, best {s['best_epoch']}, {s['seconds']}s)")
        assert len(np.unique(y_pred)) > 1, f"{kind} fold {k} collapsed to one class"

        # Written after every fold so partial results survive an interrupted run.
        json.dump(results, open(RESULTS_DIR / f"cv_partial_{kind}.json", "w"), indent=2)
    return results


cv = {}
for kind in ("lstm", "cnn"):
    log(f"=== {kind} cross-validation ===")
    cv[kind] = run_cv(kind)
log("cross-validation complete")

In [ ]:
summary = pd.DataFrame([
    {"model": kind, "metric": m,
     "mean": np.mean([f[m] for f in folds]), "std": np.std([f[m] for f in folds]),
     "min": np.min([f[m] for f in folds]), "max": np.max([f[m] for f in folds])}
    for kind, folds in cv.items()
    for m in ("accuracy", "precision_macro", "recall_macro", "f1_macro")
]).round(4)
summary.to_csv(RESULTS_DIR / "cv_summary.csv", index=False)
summary.pivot(index="metric", columns="model", values=["mean", "std"]).round(3)

In [ ]:
# Is the gap bigger than the noise? Paired across folds, since both models saw
# identical fold splits.
acc = {k: np.array([f["accuracy"] for f in v]) for k, v in cv.items()}
diff = acc["cnn"] - acc["lstm"]
baseline = holdout_df["composer"].value_counts().max() / len(holdout_df)

for k in acc:
    log(f"{k} per-fold accuracy {np.round(acc[k], 3)} "
        f"mean {acc[k].mean():.3f} +/- {acc[k].std():.3f}")
log(f"cnn - lstm per fold: {np.round(diff, 3)}, mean {diff.mean():.3f} "
    f"(cnn wins {int((diff > 0).sum())}/{N_FOLDS})")
log(f"majority-class baseline: {baseline:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(N_FOLDS)
for kind in cv:
    ax[0].plot(x, acc[kind], "o-", label=kind.upper())
ax[0].axhline(baseline, ls="--", c="grey", label="majority baseline")
ax[0].set(xticks=x, xlabel="fold", ylabel="accuracy", ylim=(0, 1),
          title="Accuracy per fold")
ax[0].legend()

width = 0.35
for i, kind in enumerate(cv):
    means = [np.mean([f["per_class"][c]["f1"] for f in cv[kind]]) for c in COMPOSERS]
    errs = [np.std([f["per_class"][c]["f1"] for f in cv[kind]]) for c in COMPOSERS]
    ax[1].bar(np.arange(4) + (i - 0.5) * width, means, width, yerr=errs,
              capsize=4, label=kind.upper())
ax[1].set(xticks=range(4), xticklabels=COMPOSERS, ylabel="F1", ylim=(0, 1),
          title="Per-composer F1 (mean +/- sd over folds)")
ax[1].legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "cv_folds.png", dpi=120)
plt.show()

## Final models and the holdout

The holdout has not been read by anything above this cell.

Each final model trains on **all** fold data — no validation slice held back — for the median number of epochs its CV runs found best. Using a validation split here would throw away a fifth of the training data, and the epoch count is the only thing that split would have decided. Then one prediction pass on the holdout, per model, and that is the number for the report.

In [ ]:
final = {}
for kind in ("lstm", "cnn"):
    n_epochs = int(np.median([f["best_epoch"] for f in cv[kind]]))
    X_tr, y_tr = build_arrays(folds_df, kind, augment=True)
    X_ho, y_ho = build_arrays(holdout_df, kind, augment=False)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = BUILD[kind]()
    log(f"=== final {kind}: {len(y_tr)} rows, {n_epochs} epochs ===")
    model.fit(X_tr, y_tr, epochs=n_epochs, batch_size=BATCH,
              class_weight=class_weights_for(folds_df),
              callbacks=[LogProgress(f"final {kind}", n_epochs)], verbose=0)

    y_pred = model.predict(X_ho, verbose=0).argmax(axis=1)
    s = score(y_ho, y_pred)
    s["epochs"] = n_epochs
    final[kind] = s
    log(f"{kind} HOLDOUT: acc {s['accuracy']:.3f} macro-F1 {s['f1_macro']:.3f}")
    assert len(np.unique(y_pred)) > 1, f"final {kind} collapsed to one class"

json.dump({"cv": cv, "holdout": final,
           "config": {"folds": N_FOLDS, "epochs_cv": EPOCHS_CV,
                      "patience_cv": PATIENCE_CV, "batch": BATCH, "seed": SEED}},
          open(RESULTS_DIR / "cv_metrics.json", "w"), indent=2)
log("wrote results/cv_metrics.json")

In [ ]:
rows = []
for kind in ("lstm", "cnn"):
    rows.append({
        "model": kind,
        "cv_accuracy": f"{acc[kind].mean():.3f} +/- {acc[kind].std():.3f}",
        "cv_macro_f1": f"{np.mean([f['f1_macro'] for f in cv[kind]]):.3f} "
                       f"+/- {np.std([f['f1_macro'] for f in cv[kind]]):.3f}",
        "holdout_accuracy": round(final[kind]["accuracy"], 3),
        "holdout_macro_f1": round(final[kind]["f1_macro"], 3),
        "holdout_precision_macro": round(final[kind]["precision_macro"], 3),
        "holdout_recall_macro": round(final[kind]["recall_macro"], 3),
    })
headline = pd.DataFrame(rows).set_index("model")
headline.to_csv(RESULTS_DIR / "cv_headline.csv")
print(f"majority-class baseline on the holdout: {baseline:.3f}")
headline

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, kind in zip(axes, ("lstm", "cnn")):
    cm = np.asarray(final[kind]["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    ax.set(xticks=range(4), yticks=range(4), xticklabels=COMPOSERS,
           yticklabels=COMPOSERS, xlabel="predicted", ylabel="true",
           title=f"{kind.upper()} holdout confusion")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    thresh = cm.max() / 2
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "confusion_holdout.png", dpi=120)
plt.show()

## Notes

- These numbers supersede the single-split ones in `results/comparison.csv` for anything claimed in the report. The single-split run stays in the repo because the gap between the two is itself the evidence for how much the leaky split was worth.
- Early stopping inside CV uses the scored fold; noted above, applies to both models equally.
- The holdout is 234 files with 19 chopin. Even a clean single number on it is noisy for the minority classes — quote the CV mean and standard deviation for per-composer claims, and the holdout for the headline.
- `results/cv_progress.log` holds the per-epoch trace of the last run, and `results/cv_partial_<model>.json` holds per-fold results as they complete, so an interrupted run does not lose everything.